In [1]:
import os, numpy as np
from google import genai
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
EMB, GEN = "gemini-embedding-001", "gemini-2.5-flash"
 
# 1. 문서 임베딩 
documents = [
    "환불은 구매일로부터 7일 이내에만 가능합니다.",
    "배송은 결제 후 평균 2~3일이 소요됩니다.",
    "신규 회원에게는 5,000원 쿠폰을 즉시 지급합니다.",
    "고객센터는 평일 09시부터 18시까지 운영합니다.",
]
def embed(text):
    r = client.models.embed_content(model=EMB, contents=text)
    return np.array(r.embeddings[0].values)
doc_vecs = [embed(d) for d in documents]
 
# 2. 질문 임베딩 
question = "청킹에서 오버랩이 필요한 이유는 무엇인가요?"
q_vec = embed(question)
 
# 3. 코사인 유사도로 Top-K 검색 
def cos(a, b): return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
scored = sorted(zip(documents, doc_vecs), key=lambda x: -cos(q_vec, x[1]))
top_k = [doc for doc, _ in scored[:2]]
 
# 4. 프롬프트 주입 + 답변 생성
prompt = f"다음 자료만 근거로 답하라.\n자료:\n" + "\n".join(top_k) + f"\n\n질문: {question}"
print(client.models.generate_content(model=GEN, contents=prompt).text)

주어진 자료만을 근거로 판단했을 때, '청킹에서 오버랩이 필요한 이유'에 대한 내용은 포함되어 있지 않습니다.

자료는 고객센터 운영 시간과 환불 가능 기간에 대한 정보만을 제공합니다.


In [2]:
import numpy as np
import ollama

EMB = "nomic-embed-text"
GEN = "llama3.2:3b"

# 1. 문서 임베딩
documents = [
    "환불은 구매일로부터 7일 이내에만 가능합니다.",
    "배송은 결제 후 평균 2~3일이 소요됩니다.",
    "신규 회원에게는 5,000원 쿠폰을 즉시 지급합니다.",
    "고객센터는 평일 09시부터 18시까지 운영합니다.",
]

def embed(text):
    r = ollama.embed(model=EMB, input=text)
    return np.array(r['embeddings'][0])

doc_vecs = [embed(d) for d in documents]

# 2. 질문 임베딩
question = "청킹 전략 방식 4가지는 무엇인가요?"
q_vec = embed(question)

# 3. 코사인 유사도로 Top-K 검색
def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

scored = sorted(zip(documents, doc_vecs), key=lambda x: -cos(q_vec, x[1]))
top_k = [doc for doc, _ in scored[:2]]

# 4. 프롬프트 주입 + 답변 생성
prompt = f"다음 자료만 근거로 답하라.\n자료:\n" + "\n".join(top_k) + f"\n\n질문: {question}"
response = ollama.chat(
    model=GEN,
    messages=[{"role": "user", "content": prompt}]
)
print(response['message']['content'])

다음 자료를 근거로하여 câu hỏi의 답변을 하겠습니다.

물론이니, 청킹 전략은 다양한 방식을 사용하지만, 일반적으로 4가지로 나눌 수 있습니다:

1. **아카이드**: 아카이드는 구매 후 지연된 배송으로 인해 환불을 받지 못하거나, 환불시기까지 시간이 طول면 청킹을 하여 부상금을 얻는 방식입니다.
2. **리턴**: 리턴은 환불을 받기 위해 제품을 다시 returning 해주는 방식입니다. 이때, 배송과 환불시기가 길어도 리턴을 할 수 있습니다.
3. **인사전환**: 인사전환은 청キング 전략의 일종으로, 청킹이 지연될 경우 원래 제품에 대한 환불을 request하고, 다시 청킹하여 부상금을 얻는 방식입니다.
4. **리메이드**: 리메イド는 청킹 후, 다시 청 kingdoms를 통해 부상금을 받는 방식입니다.

다음은 청キング 전략 4가지의 예시입니다:

* 아카이드: 제품을 구입하고 배송이 지연되면, 환불을 request하고, 다시 청キング하여 부상금을 얻습니다.
* 리턴: 제품을 구입하고 배송이 지연되면, 제품을 다시 returning해주고, 원래의 제품에 대한 환불을 request합니다.
* 인사전환: 청킹이 지연되면, 원래의 제품에 대한 환불을 request하고, 다시 청 kingdoms를 통해 부상금을 얻습니다.
* 리메이드: 청キング 후, 다시 청 kingdoms를 통해 부상금을 받습니다.

다음은 청キング 전략 4가지의 예시입니다:

* 아카이드: 제품을 구입하고 배송이 지연되면, 환불을 request하고, 다시 청 kingdoms를 통해 부상금을 얻습니다.
* 리턴: 제품을 구입하고 배송이 지연되면, 제품을 다시 returning해주고, 원래의 제품에 대한 환불을 request합니다.
* 인사전환: 청킹이 지연되면, 원래의 제품에 대한 환불을 request하고, 다시 청 kingdoms를 통해 부상금을 얻습니다.
* 리메이드: 청キング 후, 다시 청 kingdoms를 통해 부상금을 받습니다.

다음은 청キング 전략 4가지의 예시입니다: